# EDA — 250m 생활인구 데이터 (2026-10-29)

원본 데이터를 독립적으로 점검·정리합니다. 원본 파일은 수정하지 않으며, 흡연 정답 라벨이나 다른 데이터와의 결합은 만들지 않습니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from zipfile import ZipFile

# Drive에서 복사한 data/raw_data 아래의 CSV 또는 CSV가 포함된 ZIP을 찾습니다.
def find_population_source(raw_root, csv_filename):
    if not raw_root.is_dir():
        raise FileNotFoundError(f"원본 데이터 폴더가 없습니다: {raw_root}")

    csv_candidates = sorted(p for p in raw_root.rglob(csv_filename) if p.is_file())
    if csv_candidates:
        return csv_candidates[0], None

    zip_candidates = sorted(
        p for p in raw_root.rglob("*.zip")
        if p.is_file() and csv_filename.removesuffix(".csv") in p.name
    )
    for zip_path in zip_candidates:
        with ZipFile(zip_path) as archive:
            members = [
                member for member in archive.namelist()
                if not member.endswith("/") and Path(member).name == csv_filename
            ]
            if members:
                return zip_path, sorted(members)[0]

    raise FileNotFoundError(
        f"원본 파일을 찾지 못했습니다: {raw_root} 하위의 {csv_filename} 또는 "
        f"{csv_filename.removesuffix('.csv')}가 포함된 ZIP 파일을 확인하세요."
    )


def read_population_csv(source_path, zip_member, **kwargs):
    if zip_member is None:
        return pd.read_csv(source_path, **kwargs)
    with ZipFile(source_path) as archive:
        with archive.open(zip_member) as stream:
            return pd.read_csv(stream, **kwargs)

# VS Code/Jupyter 실행 위치와 관계없이 GitHub 저장소의 상위 data를 사용합니다.
def find_repo():
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if p.name == "do_not_lung" and (p / "notebooks").is_dir():
            return p
        candidate = p / "do_not_lung"
        if (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("do_not_lung 저장소를 찾지 못했습니다.")

REPO = find_repo()
DATA_ROOT = REPO.parent / "data"
RAW_ROOT = DATA_ROOT / "raw_data"
raw_path, raw_member = find_population_source(RAW_ROOT, "250_LOCAL_RESD_20261029.csv")
out_dir = DATA_ROOT / "processed_data" / "local_resd_250m_20261029"
out_dir.mkdir(parents=True, exist_ok=True)

raw = read_population_csv(raw_path, raw_member, encoding="cp949", dtype=str)
print("원본 크기:", raw.shape)
display(raw.head())


In [ ]:
id_cols = ["일자", "시간", "행정동코드", "250M격자"]
age_cols = [c for c in raw.columns if c.startswith("남자 ") or c.startswith("여자 ")]
clean = raw.copy()
clean["일자"] = pd.to_datetime(clean["일자"].str.strip(), format="%Y%m%d", errors="coerce")
clean["시간"] = pd.to_numeric(clean["시간"].str.strip(), errors="coerce").astype("Int64")
for c in ["행정동코드", "250M격자"]:
    clean[c] = clean[c].astype("string").str.strip()
clean["생활인구합계_미제공표시"] = clean["생활인구합계"].astype("string").str.strip().eq("*").fillna(False)
clean["생활인구합계"] = pd.to_numeric(clean["생활인구합계"].str.replace(",", "", regex=False).str.strip(), errors="coerce")
for c in age_cols:
    s = clean[c].astype("string").str.strip()
    clean[c + "_미제공표시"] = s.eq("*").fillna(False)
    clean[c] = pd.to_numeric(s.replace("*", pd.NA), errors="coerce")

clean["일자_유효"] = clean["일자"].notna()
clean["시간_유효"] = clean["시간"].between(0, 23).fillna(False)
clean["행정동코드_형식유효"] = clean["행정동코드"].str.fullmatch(r"\\d{8}", na=False)
clean["격자코드_형식유효"] = clean["250M격자"].str.fullmatch(r"[가-힣]{2}[0-9]{8}", na=False)
clean["생활인구합계_결측"] = clean["생활인구합계"].isna()
# 원본 날짜·행정동·격자 코드 형식은 모델의 공간 결합에 직접 사용됩니다.
assert clean["일자"].notna().all(), "일자 파싱 오류"
assert clean["시간"].between(0, 23).fillna(False).all(), "시간대 범위 오류"

male_cols = [c for c in age_cols if c.startswith("남자 ")]
female_cols = [c for c in age_cols if c.startswith("여자 ")]
clean["남자_연령합계"] = clean[male_cols].sum(axis=1, min_count=1)
clean["여자_연령합계"] = clean[female_cols].sum(axis=1, min_count=1)
clean["연령별_인구합계"] = clean[age_cols].sum(axis=1, min_count=1)
clean["남녀_연령합계차이"] = clean["남자_연령합계"] - clean["여자_연령합계"]
clean["생활인구합계와_연령합계차이"] = clean["생활인구합계"] - clean["연령별_인구합계"]
clean["시간대구분"] = pd.cut(clean["시간"].astype(float), bins=[-1, 5, 11, 17, 23], labels=["심야", "오전", "오후", "저녁"])
clean["주야구분"] = np.where(clean["시간"].between(6, 17), "주간", np.where(clean["시간"].between(18, 23) | clean["시간"].between(0, 5), "야간", pd.NA))

print("정확 중복:", raw.duplicated().sum())
print("복합키 중복:", raw.duplicated(id_cols).sum())
print("연령별 * 표시 셀:", int(clean[[c + "_미제공표시" for c in age_cols]].to_numpy().sum()))
display(clean.describe(include="all").T.head(15))


In [ ]:
clean_export = clean.copy()
clean_export["일자"] = clean_export["일자"].dt.strftime("%Y-%m-%d")
clean_export.to_csv(out_dir / "clean.csv", index=False, encoding="utf-8-sig")

hour_summary = clean.groupby("시간", dropna=False).agg(
    행수=("생활인구합계", "size"),
    생활인구합계_평균=("생활인구합계", "mean"),
    생활인구합계_중앙값=("생활인구합계", "median"),
    생활인구합계_합계=("생활인구합계", "sum")
).reset_index()
dong_summary = clean.groupby("행정동코드", dropna=False).agg(
    행수=("생활인구합계", "size"), 격자수=("250M격자", "nunique"),
    생활인구합계_평균=("생활인구합계", "mean"), 생활인구합계_합계=("생활인구합계", "sum")
).reset_index()
grid_summary = clean.groupby("250M격자", dropna=False).agg(
    행수=("생활인구합계", "size"), 행정동수=("행정동코드", "nunique"),
    생활인구합계_평균=("생활인구합계", "mean"), 생활인구합계_최대=("생활인구합계", "max")
).reset_index()
quality = pd.DataFrame([
    ["원본 행 수", len(raw)], ["원본 열 수", len(raw.columns)],
    ["정확 중복 행 수", int(raw.duplicated().sum())],
    ["복합키 중복 행 수", int(raw.duplicated(id_cols).sum())],
    ["고유 일자 수", raw["일자"].nunique()], ["고유 시간 수", raw["시간"].nunique()],
    ["고유 행정동 수", raw["행정동코드"].nunique()], ["고유 250m 격자 수", raw["250M격자"].nunique()],
    ["생활인구합계 결측 수", int(clean["생활인구합계"].isna().sum())],
    ["연령별 * 표시 셀 수", int(clean[[c + "_미제공표시" for c in age_cols]].to_numpy().sum())]
], columns=["항목", "값"])
parts = []
for group_name, frame in [("시간대별", hour_summary), ("행정동별", dong_summary), ("격자별", grid_summary)]:
    for _, row in frame.iterrows():
        parts.append({"구분": group_name, **row.to_dict()})
for _, row in quality.iterrows():
    parts.append({"구분": "품질점검", "항목": row["항목"], "값": row["값"]})
summary = pd.DataFrame(parts)
summary.to_csv(out_dir / "summary.csv", index=False, encoding="utf-8-sig")

feature_cols = id_cols + ["생활인구합계"] + age_cols + [
    "남자_연령합계", "여자_연령합계", "연령별_인구합계",
    "남녀_연령합계차이", "생활인구합계와_연령합계차이", "시간대구분", "주야구분"
]
features = clean_export[feature_cols]
features.to_csv(out_dir / "features.csv", index=False, encoding="utf-8-sig")
# 다른 EDA와 동일하게 저장 결과를 다시 읽어 컬럼·행 수·식별자를 검증합니다.
for filename, frame in [("clean.csv", clean_export), ("summary.csv", summary), ("features.csv", features)]:
    saved = out_dir / filename
    assert saved.read_bytes().startswith(b"\\xef\\xbb\\xbf"), filename
    reread = pd.read_csv(saved, encoding="utf-8-sig", dtype={"행정동코드": "string", "250M격자": "string"})
    assert reread.shape == frame.shape, filename
    assert list(reread.columns) == list(frame.columns), filename
    if filename in ("clean.csv", "features.csv"):
        assert reread["행정동코드"].fillna("").tolist() == frame["행정동코드"].fillna("").astype(str).tolist(), filename
        assert reread["250M격자"].fillna("").tolist() == frame["250M격자"].fillna("").astype(str).tolist(), filename

print("clean:", clean_export.shape, "summary:", summary.shape, "features:", features.shape)
print("CSV 저장·재읽기 검증 완료")


In [ ]:
# 시간대별 평균 생활인구 분포
hour_summary.plot(x="시간", y="생활인구합계_평균", kind="line", marker="o", figsize=(10, 4))
plt.title("2026-10-29 시간대별 평균 생활인구")
plt.xlabel("시간"); plt.ylabel("평균 생활인구"); plt.grid(alpha=.3); plt.show()

# 생활인구 합계 분포
clean["생활인구합계"].dropna().plot(kind="hist", bins=40, figsize=(8, 4))
plt.title("생활인구합계 분포"); plt.xlabel("생활인구합계"); plt.ylabel("행 수"); plt.show()


## 해석 시 주의
- 하루 자료이므로 날짜 간 변화나 추세를 판단할 수 없습니다.
- 생활인구는 흡연 발생 또는 간접흡연 노출의 정답값이 아닙니다.
- 원본 `*` 표시는 의미를 확인할 메타데이터가 없어 0으로 바꾸지 않고 결측 및 표시 플래그로 보존합니다.
- 격자 코드는 식별자이며 좌표를 임의로 만들지 않습니다.
- 다른 데이터셋과 병합하거나 흡연 위험도·예측 라벨을 생성하지 않습니다.
